Control flow/goto-based Abstract Syntax Tree (AST)
--------------------------------------------------

Pre-condition Required Transforms
---------------------------------

Expressions and declarator initalizers containing operators with side-effects in sequence point order (respecting ",", &&, ||, function call () operators): transform(S, Z) puts statement S as new statement before expression and replaces expression with newly introduced variable Z.  This is done recursively on expressions appearing in S.

Assignment (possible side effect of segmentation fault, modifying memory contents): =, +=, -=, *=, /=, %=, &=, |=, ^=, <<=, >>=, prefix ++, prefix --, postfix ++, postfix --

Function calls (arguments are evaluated, possible modifying of memory contents, faults)

Pointer dereference (possible side effect of segmentation fault): *, [], ->

Division/modulo (possible side effect of division by zero fault): /, %

X OP Y -> transform(Z = X OP Y, Z) 

X && Y -> transform(switch (X) { case 1: if (Y) {Z=1; break; } case 0: Z=0 }, Z)

X || Y -> transform(switch (X) { case 0: if (!Y) {Z=0; break; } case 0: Z=1 }, Z)

cond ? X : Y -> transform(if (cond) { Z=X; } else { Z=Y; }, Z)


for (...) { seq } -> for (...) { seq HEADLBL: } NEXTLBL:

while (...) { seq } -> while (...) { seq HEADLBL: } NEXTLBL:

do { seq } while (...); -> do { seq HEADLBL: } while (...); NEXTLBL:

do { seq } while (cond); -> while (true) { seq if (!cond) break; }

switch (mcond) { ... } -> switch (mcond) { ... } NEXTLBL:

break; -> goto switch/while/do-while/inf-loop NEXTLBL;

continue; -> goto while/do-while/inf-loop HEADLBL;


if (cond) {} else { seq } -> if (!cond) { seq }

if (cond) { seq } else {} -> if (cond) { seq }

switch (mcond) { ... (default:|case .+:)+ } -> switch (mcond) { ... } where + is 1 or more, . is any character

if (cond) {} -> $\emptyset$

switch (mcond) {} -> $\emptyset$

while (false) { ... } -> $\emptyset$

switch (mcond) { case X: seq } -> if (mcond == X) { seq }

switch (mcond) { default: seq } -> seq

for (init; cond; inc) { ... } -> { init; while (cond) { ... inc; } } 

switch (mcond) { case X: seq1 default: seq2 } -> if (mcond == X) { seq1 } else { seq2 }

do { seq } while (false); -> seq

{} -> $\emptyset$


AST Elements
------------
Elements are sequence or codeblock or control-flow statement

Head -> (Data, Children)

Children is [First, ..., Last]

sequence is a maximal non-empty statement block

sequence sequence -> sequence $\mathbin\Vert$ sequence where $\mathbin\Vert$ is concatenation

empty sequence -> $\emptyset$

codeblock is a statement block with no control-flow statements

if/if-else/switch/while/do-while/inf-loop/goto/return are control-flow statements

Return is a special node indicating return from a function

Control-Flow Elements with Top-Down Algorithm Terminating, Next and Graph Edges
-------------------------------------------------------------------------------
| Element   | Data                    | Children | Terminating | Next[Children]           | Graph Edges      |
| --------- | ----------------------- | -------- | ----------- | ------------------------ | ---------------- |
| sequence  | -                       | Yes (>0) | Yes         | Children[1:] $\mathbin\Vert$ Next[Head] | Head -> First    |
| codeblock | Code                    | No       | No          | -                        | -                |
| if        | Condition               | 1        | No          | [Next[Head]]             | Head -> First    |
| if-else   | Condition               | 2        | Yes         | [Next[Head], Next[Head]] | Head -> Children |
| switch    | Selector, [Value-list]  | Yes (>1) | Yes         | Children[1:] $\mathbin\Vert$ Next[Head] | Head -> Children |
| while     | Condition               | 1        | No          | [Head]                   | Head -> First    |
| do-while  | Condition               | 1        | No          | [Head]                   | Head -> First    |
| inf-loop  | -                       | 1        | Yes         | [Head]                   | Head -> First    |
| goto      | Target                  | No       | Yes         | -                        | Head -> Target   |
| return    | optional Expression     | No       | Yes         | -                        | Head -> Return   |

Remaining graph edges:
if (Head $\notin$ Terminating) {
    if type(Head) is do-while and Last $\notin$ Terminating { Last -> Next[Head] }
    else Head -> Next[Head]
}

Eliminate dead-code by reachability from root in CFG (either by BFS or DFS)

switch/while/do-while/inf-loop stack (Head is top of stack): goto Next[Head] -> break;

while/do-while/inf-loop stack (Head is top of stack): goto Head -> continue;

Hamiltonian cycle problem:
1) Choose any node as root node
2) Check reachability from source node with BFS or DFS, if not all nodes reachable return No
3) Digraph -> Control Flow Graph with chosen root
4) Hamiltonian cycle is execution cycle of CFG which executes each code block once and only once before reaching the root node again

Maximum tree edges in incremental DFS:
Base case: Root -> Node0
Root -> Node1 and then Node0 -> Node1
Node0 -> Node

